In [5]:
import pandas as pd

df = pd.read_csv("bdc_37_fixed_broadband_served_unserved_J26_24oct2026.csv")
county_geoids_df = pd.read_csv("ncstate_county_geoids.csv")

In [6]:
df = df.astype({"location_id": str,"block_geoid": str, "any_dl100_ul20": bool, "wired_dl100_ul20": bool, "terrestrial_dl100_ul20": bool})

In [7]:
df["County GEOID"] = df["block_geoid"].str[2:5]
df["Census Tract GEOID"] = df["block_geoid"].str[5:11]

In [8]:
clean_df = df[["County GEOID", "Census Tract GEOID", "any_dl100_ul20", "wired_dl100_ul20", "terrestrial_dl100_ul20"]]

In [9]:
county_geoids_df = county_geoids_df.astype({"FIPS Code": str})
county_geoids_df["FIPS Code"] = county_geoids_df["FIPS Code"].str[2:]
county_geoids_df["County"] = county_geoids_df["County"].str.replace(" County", "").str.strip()

In [10]:
county_geoids_df.head()

,County,FIPS Code,Tracts
0,Alamance,001,36
1,Alexander,003,7
2,Alleghany,005,3
3,Anson,007,6
4,Ashe,009,6


In [11]:
clean_df.head()

,County GEOID,Census Tract GEOID,any_dl100_ul20,wired_dl100_ul20,terrestrial_dl100_ul20
0,171,930301,True,False,True
1,189,920800,True,True,True
2,189,920900,True,True,True
3,189,920702,True,True,True
4,189,920200,True,True,True


In [12]:
merged_df = pd.merge(clean_df, county_geoids_df[["County", "FIPS Code"]], left_on="County GEOID", right_on="FIPS Code", how="left")
county_col = merged_df.pop("County")
merged_df = merged_df.drop(columns=["County GEOID", "FIPS Code"])
merged_df.insert(0, "County", county_col)

In [13]:
merged_df.head()

,County,Census Tract GEOID,any_dl100_ul20,wired_dl100_ul20,terrestrial_dl100_ul20
0,Surry,930301,True,False,True
1,Watauga,920800,True,True,True
2,Watauga,920900,True,True,True
3,Watauga,920702,True,True,True
4,Watauga,920200,True,True,True


In [14]:
merged_df["no_terra_service"] = merged_df["terrestrial_dl100_ul20"] == False
by_county = merged_df.groupby("County")["no_terra_service"].agg(["size","sum","mean"])
by_county = by_county.sort_values(by="mean", ascending=False)
by_county["% Without Coverage"] = round(by_county["mean"] * 100,2)
by_county = by_county.rename(columns={"size": "Total Locations", "sum": "No Terrestrial Service"})
by_county.drop(columns="mean")

,Total Locations,No Terrestrial Service,% Without Coverage
County,,,
Graham,5507,2401,43.60
Swain,10399,4292,41.27
Clay,8106,2646,32.64
Jackson,25809,7274,28.18
Hyde,3824,1076,28.14
...,...,...,...
Davie,20358,39,0.19
Brunswick,106119,193,0.18
New Hanover,91798,150,0.16


In [29]:
threshold = 15

by_tract = merged_df.groupby(["County", "Census Tract GEOID"])["no_terra_service"].agg(["size","sum","mean"])
by_tract = by_tract.sort_values(by="mean", ascending=False)
by_tract["% Without Coverage"] = round(by_tract["mean"] * 100,2)
by_tract = by_tract.rename(columns={"size": "Total Locations", "sum": "No Terrestrial Service"})
by_tract = by_tract.drop(columns="mean")
by_tract = by_tract[by_tract["Total Locations"] >= 100]
by_tract = by_tract[by_tract["% Without Coverage"] >= threshold]
print(len(by_tract))
by_tract.head()

120


Total Locations  No Terrestrial Service  \
County  Census Tract GEOID                                            
Jackson 950800                         2987                    2234   
        950700                         1911                    1427   
Swain   960301                         2807                    1995   
Graham  920100                         2057                    1460   
Clay    950102                         1973                    1162   

                            % Without Coverage  
County  Census Tract GEOID                      
Jackson 950800                           74.79  
        950700                           74.67  
Swain   960301                           71.07  
Graham  920100                           70.98  
Clay    950102                           58.90